In [0]:
%run ./00_config

In [0]:
# dbutils.fs.ls(BRONZE_PATH)

In [0]:
# files = dbutils.fs.ls(BRONZE_PATH)
# newest = sorted(files, key=lambda f: f.name)[-1]
# print("Newest folder:", newest.path)

# inside = dbutils.fs.ls(newest.path)
# for f in inside:
#     print(f.name, f.size)
    

In [0]:
#dbutils.fs.ls("/Volumes/workspace/default/university_chapters/bronze/20260913T153038Z_e8ede8fd/")

In [0]:
import json

bronze_run_path = latest_bronze_run_path()
bronze_file = f"{bronze_run_path}/raw_response.json"

with open(bronze_file, "r") as f:
    bronze_record = json.load(f)

print(f"Reading bronze from: {bronze_file}")

In [0]:
features = bronze_record["raw_response"].get("features", [])
print(f"Number of feature rows: {len(features)}")

In [0]:
flat_rows = []

for feature in features:
    attributes = feature.get("attributes", {})
    geometry = feature.get("geometry", {})

    flat_row = dict(attributes)  # copy attributes into a new flat dict
    flat_row["x"] = geometry.get("x")
    flat_row["y"] = geometry.get("y")

    flat_rows.append(flat_row)

print(flat_rows[0])  # peek at the first row to sanity-check

In [0]:
df_raw = spark.createDataFrame(flat_rows)
# df_raw.printSchema()
# #df_raw.show(truncate=False)
# df_raw.display()

In [0]:
#translate the raw API column names into our published names
df_renamed = df_raw

for source_col, target_col in COLUMN_MAPPING.items():
    df_renamed = df_renamed.withColumnRenamed(source_col, target_col)

df_renamed = df_renamed.withColumnRenamed("x", "longitude").withColumnRenamed("y", "latitude")

df_renamed.printSchema()

In [0]:
#injecting a synthetic bad row
synthetic_rows = spark.createDataFrame([
    {
        "chapter_id": "TEST-BAD-001",
        "chapter_name": "Synthetic Test Chapter",
        "city": "Test City",
        "state": "CA",
        "source_object_id": -1,
        "MEVR_RD": "ignore",
        "longitude": 200.0,   # invalid: outside [-180, 180]
        "latitude": 37.0,
    },
    {
        "chapter_id": "TEST-WARN-001",
        "chapter_name": "Synthetic Warning Chapter",
        "city": "UNKNOWN",   # triggers DQ-W1
        "state": "OR",
        "source_object_id": -2,
        "MEVR_RD": "ignore",
        "longitude": -120.5,   # valid coordinates
        "latitude": 44.0,
    },
])

df_with_test_row = df_renamed.unionByName(synthetic_rows)

df_with_test_row.select("chapter_id", "city", "longitude", "latitude").show()

In [0]:
from pyspark.sql import functions as F

invalid_coords_condition = (
    F.col("longitude").isNull() | F.col("latitude").isNull() |
    (F.col("longitude") < -180) | (F.col("longitude") > 180) |
    (F.col("latitude") < -90) | (F.col("latitude") > 90)
)

df_flagged = df_with_test_row.withColumn("is_invalid_coords", invalid_coords_condition)

df_flagged.select("chapter_id", "longitude", "latitude", "is_invalid_coords").show()

In [0]:
#split into Quarantine vs. continuing rows, and write Quarantine output
df_quarantine = df_flagged.filter(F.col("is_invalid_coords") == True)
df_clean_coords = df_flagged.filter(F.col("is_invalid_coords") == False)

print(f"Rows to quarantine: {df_quarantine.count()}")
print(f"Rows continuing to Silver: {df_clean_coords.count()}")

In [0]:
#add the required quarantine metadata
df_quarantine_final = df_quarantine.withColumn(
    "reason_code", F.lit(DQ_REASON_INVALID_COORDINATES)
).withColumn(
    "ingest_run_id", F.lit(bronze_record["ingest_run_id"])
)

df_quarantine_final.select("chapter_id", "longitude", "latitude", "reason_code", "ingest_run_id")

In [0]:
quarantine_run_folder = f"{QUARANTINE_PATH}/{bronze_record['ingest_run_id']}"

df_quarantine_final.write.mode("overwrite").format("delta").save(quarantine_run_folder)

print(f"Wrote quarantine data to: {quarantine_run_folder}")

In [0]:
dbutils.fs.ls(quarantine_run_folder)

In [0]:
city_warning_condition = (
    F.col("city").isNull() |
    (F.trim(F.col("city")) == "") |
    (F.upper(F.trim(F.col("city"))) == "UNKNOWN")
)

df_with_warning_flag = df_clean_coords.withColumn("has_city_warning", city_warning_condition)

df_with_warning_flag.select("chapter_id", "city", "has_city_warning").show()

In [0]:
df_silver = df_with_warning_flag.withColumn(
    "dq_status",
    F.when(F.col("has_city_warning") == True, F.lit(DQ_STATUS_WARNING))
     .otherwise(F.lit(DQ_STATUS_OK))
).withColumn(
    "dq_warnings",
    F.when(F.col("has_city_warning") == True, F.lit(DQ_REASON_MISSING_UNKNOWN_CITY))
     .otherwise(F.lit(""))
).withColumn(
    "ingest_run_id", F.lit(bronze_record["ingest_run_id"])
)

df_silver.select("chapter_id", "city", "dq_status", "dq_warnings").show()

In [0]:
df_silver_deduped = df_silver.dropDuplicates(["chapter_id"])

print(f"Rows before dedup: {df_silver.count()}")
print(f"Rows after dedup: {df_silver_deduped.count()}")

In [0]:
df_silver_deduped.write.mode("overwrite").format("delta").save(SILVER_PATH)

In [0]:
rows_in = len(features)  # from notebook 1's data, re-derived here for clarity
rows_quarantined = df_quarantine_final.count()
rows_warned = df_silver_deduped.filter(F.col("dq_status") == DQ_STATUS_WARNING).count()
rows_ok = df_silver_deduped.filter(F.col("dq_status") == DQ_STATUS_OK).count()

print(f"Run ID: {bronze_record['ingest_run_id']}")
print(f"rows_in: {rows_in}")
print(f"rows_quarantined: {rows_quarantined}")
print(f"rows_warned: {rows_warned}")
print(f"rows_ok: {rows_ok}")